# Assignment 2

 **This was supoosed to be 2 assignment but is a big assignment So go slow and learn what you are doing have fun**


# Part 1: Data Ingestion

Data Ingestion is the first step in a RAG pipeline. It involves collecting, reading, and loading raw data from various sources (such as PDFs, HTML, or databases) into the system. Here, we read all PDF files in a given directory and parse their content into structured documents containing page content and metadata.


Here we Doing only for pdf but in final project we will do for pdf,csv,xlsx,docx,txt.
if you want you can practise to extract data from one more file format i would love to see you do this.

In [1]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


C:\Users\gunja\AppData\Local\Temp\ipykernel_41760\2076300660.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
!pip install -U langchain-community

In [3]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    from pathlib import Path
    from langchain_community.document_loaders import PyPDFLoader

    documents = []
    for pdf_file in Path(pdf_directory).glob('**/*.pdf'):
        loader = PyPDFLoader(str(pdf_file))
        docs = loader.load()

        for doc in docs:
            doc.metadata['source_file'] = pdf_file.name
            doc.metadata['file_type'] = 'pdf'

        documents.extend(docs)

    print(f"Loaded {len(documents)} pages")
    return documents


In [6]:
!pip install pypdf

In [7]:
pdf_directory = "."

documents = process_all_pdfs(pdf_directory)

print(f"Loaded {len(documents)} pages")

Loaded 38 pages
Loaded 38 pages


In [ ]:
all_pdf_documents

# Part 2: Chunking

Chunking is the process of breaking down large documents into smaller, cohesive segments (chunks). Since Large Language Models (LLMs) have a limited context window (input size limit) and retrieve information more accurately from smaller context blocks, we must split large documents. In this assignment, you need to use **RecursiveCharacterTextSplitter** to split loaded documents into smaller paragraphs with proper overlap to maintain context between boundary lines.


In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [23]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n', '\n', ' ', '']
    )

    chunks = splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(chunks)} chunks")
    return chunks


In [24]:
chunks=split_documents(documents)
chunks

Split 38 documents into 93 chunks


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-22T08:02:55+00:00', 'author': 'Based on the Vizuara Lecture Series by Dr. Raj Dandekar', 'keywords': '', 'moddate': '2026-04-22T08:02:55+00:00', 'subject': 'LLM Education', 'title': 'Building Large Language Models from Scratch', 'trapped': '/False', 'source': 'Building_LLMs_from_Scratch.pdf', 'total_pages': 38, 'page': 0, 'page_label': '1', 'source_file': 'Building_LLMs_from_Scratch.pdf', 'file_type': 'pdf'}, page_content='Building Large Language\n Models from Scratch\n A Complete Guide from Tokenization to Fine-Tuning\n Based on the Vizuara Lecture Series by Dr. Raj Dandekar\n Version 1.0 · April 2026\n Source: Vizuara YouTube Playlist — Building LLMs from Scratch (43 videos)\n Instructor: Dr. Raj Dandekar, MIT PhD\nCompanion book: Build a Large Language Model (From Scratch) — Sebastian Raschka\n (Manning, 2024)'),
 Document(metadata={'producer': 'ReportLab PDF 

# Part 3: Embedding

Embedding is the process of converting text blocks into numerical vector representations. These vectors capture the semantic meaning of the text. Sentences that are semantically similar will be closer to each other in the vector space. We use pre-trained sentence transformer models (like 'all-MiniLM-L6-v2') to convert text chunks and queries into embeddings.

---



from sentence_transformers import SentenceTransformer

Imports the embedding model class.

This library:

downloads pretrained transformer models
converts text → embeddings

Built on top of:

HuggingFace transformers
PyTorch

In [30]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid #to get each chunk a unique id after embedding
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [31]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        from sentence_transformers import SentenceTransformer
        print(f"Loading embedding model: {self.model_name}")
        self.model = SentenceTransformer(self.model_name)
        print("Model loaded successfully")
        print("Embedding dimension:", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, texts):
        if self.model is None:
            raise ValueError("Embedding model not loaded")
        return self.model.encode(texts, show_progress_bar=True)



# Part 4: Vector DB

Vector Database (Vector DB) is a specialized database designed to store and query high-dimensional vector embeddings efficiently. It allows us to persist our document chunks along with their computed embeddings and perform fast search operations. Here, we use ChromaDB, which runs locally and stores documents persistently in a directory.

In [32]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        import os, chromadb
        os.makedirs(self.persist_directory, exist_ok=True)

        self.client = chromadb.PersistentClient(path=self.persist_directory)
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "PDF document embeddings"}
        )

    def add_documents(self, documents, embeddings):
        import uuid

        if len(documents) != len(embeddings):
            raise ValueError("Documents and embeddings count mismatch")

        ids, metas, docs, embs = [], [], [], []

        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            ids.append(uuid.uuid4().hex[:8])

            meta = dict(doc.metadata)
            meta["doc_index"] = i
            meta["content_length"] = len(doc.page_content)
            metas.append(meta)

            docs.append(doc.page_content)
            embs.append(emb.tolist())

        self.collection.add(
            ids=ids,
            embeddings=embs,
            metadatas=metas,
            documents=docs
        )


In [ ]:
chunks

## convert text to embeddings


In [33]:
### lets see text of chumks
texts=[doc.page_content for doc in chunks]

texts

['Building Large Language\n Models from Scratch\n A Complete Guide from Tokenization to Fine-Tuning\n Based on the Vizuara Lecture Series by Dr. Raj Dandekar\n Version 1.0 · April 2026\n Source: Vizuara YouTube Playlist — Building LLMs from Scratch (43 videos)\n Instructor: Dr. Raj Dandekar, MIT PhD\nCompanion book: Build a Large Language Model (From Scratch) — Sebastian Raschka\n (Manning, 2024)',
 'Table of Contents\nPart I: Foundations\n    Chapter 1: Introduction — Why Build an LLM from Scratch?\n    Chapter 2: What Are Transformers?\nPart II: Data Pre-Processing\n    Chapter 3: Tokenisation — Converting Text to Numbers\n    Chapter 4: Building the Data Pipeline\nPart III: The Attention Mechanism\n    Chapter 5: Introduction to Attention\n    Chapter 6: Self-Attention with Keys, Queries, and Values\n    Chapter 7: Multi-Head Attention\nPart IV: LLM Architecture\n    Chapter 8: Assembling the GPT Architecture\nPart V: Pretraining\n    Chapter 9: Training the LLM\nPart VI: Fine-Tunin

In [35]:
embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\gunja\.anaconda\New folder\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gunja\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully
Embedding dimension: 384


C:\Users\gunja\AppData\Local\Temp\ipykernel_41760\1664001024.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", self.model.get_sentence_embedding_dimension())


In [37]:
vectorstore = VectorStore(
    collection_name="pdf_documents"
)

In [38]:
## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

# Part 5: Query Retrieval

Query Retrieval starts with the user entering a natural language query. We must convert this query into the same embedding space using our embedding manager. This encoded query is then sent to the vector database to retrieve raw results.

---

# Part 6: Similarity Search

Similarity Search is the mathematical calculation (such as Cosine Similarity) used by the vector store to compare the query embedding with document embeddings. It ranks and returns the top_k most similar documents. We can filter results by a minimum similarity score (score_threshold) to keep only relevant context.


In [39]:
class RAGRetriever:
    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        retrieved = []

        for rank, (doc_id, doc, meta, dist) in enumerate(
            zip(results['ids'][0],
                results['documents'][0],
                results['metadatas'][0],
                results['distances'][0]), start=1):

            similarity = 1 - dist

            if similarity >= score_threshold:
                retrieved.append({
                    "id": doc_id,
                    "content": doc,
                    "metadata": meta,
                    "similarity_score": similarity,
                    "distance": dist,
                    "rank": rank
                })

        return retrieved


In [40]:
rag_retriever = RAGRetriever(
    vector_store=vectorstore,
    embedding_manager=embedding_manager
)

In [41]:
rag_retriever

In [43]:
rag_retriever.retrieve("Self-Attention with Keys, Queries, and Values")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[{'id': 'fe99b646',
  'content': 'Chapter 6: Self-Attention with Keys, Queries, and\nValues\nCovers: Lecture 14: Self-Attention with Keys, Queries and Values — Coded from Scratch · Lecture 15: Causal\nSelf-Attention — Coded from Scratch\n6.1 Why Projections? Keys, Queries, and Values\nThe simplified attention in Chapter 5 used the input vectors directly as both the query and the\nkeys. This limits expressiveness: the dot product x_i · x_j measures raw similarity in embedding\nspace, which may not be what we want to attend to. In full self-attention (as in "Attention Is All\nYou Need"), three separate projection matrices are learned:\nW_Q ∈ R^(d_model × d_k)   — Query projection\nW_K ∈ R^(d_model × d_k)   — Key projection\nW_V ∈ R^(d_model × d_v)   — Value projection\nThese transform each input vector into three separate roles:\nQ = X W_Q   — "What am I looking for?"\nK = X W_K   — "What do I contain?"\nV = X W_V   — "What do I communicate to others?"\nAttention scores are then computed

# Part 7: Prompting and Calling LLM

Prompting and Calling LLM is the synthesis step of RAG. We take the retrieved contexts, format them into a structured prompt alongside the user's original query, and pass them to the Large Language Model (LLM) to generate a grounded, factually accurate response.


In [44]:
def rag_simple(query,retriever,llm,top_k=3):
    docs = retriever.retrieve(query, top_k=top_k)

    context = "\n\n".join([d["content"] for d in docs])

    prompt = f"""Answer the question using only the context below.

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)
    return response.content if hasattr(response, "content") else str(response)


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key="",
    base_url="https://openrouter.ai/api/v1"
)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    google_api_key=""
)

Task exception was never retrieved
future: <Task finished name='Task-14' coro=<AsyncClient.aclose() done, defined at c:\Users\gunja\.anaconda\New folder\Lib\site-packages\httpx\_client.py:2011> exception=ImportError("cannot import name 'set_current_async_library' from 'anyio._core._eventloop' (c:\\Users\\gunja\\.anaconda\\New folder\\Lib\\site-packages\\anyio\\_core\\_eventloop.py)")>
Traceback (most recent call last):
  File "c:\Users\gunja\.anaconda\New folder\Lib\site-packages\anyio\_core\_eventloop.py", line 159, in get_async_backend
KeyError: 'anyio._backends._asyncio'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\gunja\.anaconda\New folder\Lib\site-packages\httpx\_client.py", line 2018, in aclose
    if proxy is not None:
  File "c:\Users\gunja\.anaconda\New folder\Lib\site-packages\httpx\_transports\default.py", line 385, in aclose
    host=request.url.raw_host,
    ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\U

In [63]:
response = llm.invoke("Hello")
print(response.content)

Hello! How can I assist you today?


In [64]:
answer = rag_simple(
    "Self-Attention with Keys, Queries, and Values",
    rag_retriever,
    llm
)

print(answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Self-Attention with Keys, Queries, and Values involves transforming input vectors into three distinct roles using learned projection matrices: Queries (Q), Keys (K), and Values (V). The projections are given by:

- Q = X W_Q (Query projection)
- K = X W_K (Key projection)
- V = X W_V (Value projection)

Attention scores are computed between queries and keys using the formula:

Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V

This approach enhances expressiveness and enables the model to attend to specific aspects of the input by computing a weighted sum of the token representations, where the weights reflect the relevance of each token to the current one.


# Part 8: Advanced RAG (Optional)

Advanced RAG includes sophisticated pipeline elements such as streaming responses, citation tracking, interaction history (conversational memory), response summarization, and score-based filtering to make the application robust and production-ready.


In [65]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    docs = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

    context = "\n\n".join([d["content"] for d in docs])

    sources = []
    for d in docs:
        sources.append({
            "source_file": d["metadata"].get("source_file","Unknown"),
            "page": d["metadata"].get("page","N/A"),
            "similarity_score": d["similarity_score"],
            "preview": d["content"][:150]
        })

    confidence = max([d["similarity_score"] for d in docs], default=0)

    prompt = f"""Use the provided context to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)
    answer = response.content if hasattr(response, "content") else str(response)

    result = {
        "answer": answer,
        "sources": sources,
        "confidence": confidence
    }

    if return_context:
        result["context"] = context

    return result


In [66]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:

        docs = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)

        context = "\n\n".join([d["content"] for d in docs])

        sources = [{
            "source_file": d["metadata"].get("source_file","Unknown"),
            "page": d["metadata"].get("page","N/A"),
            "score": d["similarity_score"]
        } for d in docs]

        prompt = f"""Use the context below to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

        if stream:
            for i in range(0, len(prompt), 200):
                print(prompt[i:i+200])
                time.sleep(0.05)

        response = self.llm.invoke(prompt)
        answer = response.content if hasattr(response, "content") else str(response)

        citations = "\n\nSources:\n" + "\n".join(
            [f"- {s['source_file']} (page {s['page']})" for s in sources]
        )

        answer_with_citations = answer + citations

        summary = None
        if summarize:
            summary_prompt = f"Summarize in 2 sentences:\n\n{answer}"
            s = self.llm.invoke(summary_prompt)
            summary = s.content if hasattr(s, "content") else str(s)

        self.history.append({
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary
        })

        return {
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary,
            "history": self.history
        }
